In [1]:
import pandas as pd
import numpy as np
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory


In [2]:
animais = ['camelo','dromedario']

quantidade = {
    'camelo':5,
    'dromedario':40,
}

quantidade_transportada = [5000,20000]

limites={
    'aveia':180,
    'agua':2500
}

informacoes = {
    'camelo': {
        'valor':60,
        'capacidade_transporte':500,
        'consumo_agua':50,
        'consumo_aveia':3
        
    },
    'dromedario':{
        'valor':25,
        'capacidade_transporte':250,
        'consumo_agua':50,
        'consumo_aveia':4
    }
}

#objetivo, CUSTO MINIMO 
# quantidade de camelo e dromedario que faz o custo minimo com as restrições


In [3]:
model = pyo.ConcreteModel()

model.animais = pyo.Set(initialize = animais)
model.x = pyo.Var(model.animais, bounds=(0,None), domain=pyo.NonNegativeIntegers)

def objetivo1(model):
    return sum(
        model.x[a] * informacoes[a]['valor'] for a in model.animais
    )
model.obj = pyo.Objective(rule=objetivo1, sense=pyo.minimize)

# restrições
def total_transportada_min(model):
    return sum(
        model.x[a] * informacoes[a]['capacidade_transporte'] for a in model.animais
    ) >= quantidade_transportada[0] 
model.minimo_transportado = pyo.Constraint(rule=total_transportada_min)
def total_transportada_max(model):
    return sum(
        model.x[a] * informacoes[a]['capacidade_transporte'] for a in model.animais
    ) <= quantidade_transportada[1] 
model.maximo_transportado = pyo.Constraint(rule=total_transportada_max)

def consumo_agua1(model):
    return sum(
        model.x[a] * informacoes[a]['consumo_agua'] for a in model.animais
    )<=limites['agua']
model.consumo_agua = pyo.Constraint( rule=consumo_agua1)
def consumo_aveia1(model):
    return sum(
        model.x[a] * informacoes[a]['consumo_aveia'] for a in model.animais
    )<=limites['aveia']
model.consumo_aveia = pyo.Constraint( rule=consumo_aveia1)

def quantidade_camelo(model, a):
    return model.x['camelo'] >= quantidade['camelo']
model.quantidade_camelo = pyo.Constraint(model.animais, rule=quantidade_camelo)
def quantidade_dromedario(model, a):
    return model.x['dromedario'] <= quantidade['dromedario']
model.quantidade_dromedario = pyo.Constraint(model.animais, rule=quantidade_dromedario)

In [4]:
model.pprint()

1 Set Declarations
    animais : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'camelo', 'dromedario'}

1 Var Declarations
    x : Size=2, Index=animais
        Key        : Lower : Value : Upper : Fixed : Stale : Domain
            camelo :     0 :  None :  None : False :  True : NonNegativeIntegers
        dromedario :     0 :  None :  None : False :  True : NonNegativeIntegers

1 Objective Declarations
    obj : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : minimize : 60*x[camelo] + 25*x[dromedario]

6 Constraint Declarations
    consumo_agua : Size=1, Index=None, Active=True
        Key  : Lower : Body                            : Upper  : Active
        None :  -Inf : 50*x[camelo] + 50*x[dromedario] : 2500.0 :   True
    consumo_aveia : Size=1, Index=None, Active=True
        Key  : Lower : Body                          : Upper : Active
        Non

In [5]:
opt = SolverFactory('cplex')
resultado = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmp38brwpms.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmpb9zz3yxj.pyomo.lp' read.
Read time = 0.02 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmpb9zz3yxj.pyomo.lp
Objective sense      : Minimize
Variables            :       2  [General Integer: 2]
Objective nonzeros   :       2
Linear constraints   :       8  [Less: 5,  Greater: 3]
  Nonzeros           :      12
  RHS nonzeros       :       8

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 25

In [10]:
for m in model.x:
    print(f'{m} = {model.x[m].value}')

print(f'Mínimo Custo: {model.obj()}')    

camelo = 5.0
dromedario = 10.0
Mínimo Custo: 550.0
